# Word Embeddings: Word2Vec (CBOW vs. Skip-Gram) and AvgWord2Vec

Train Word2Vec on a real NLTK corpus (Gutenberg), compare CBOW vs. Skip-Gram nearest neighbours, then build Average Word2Vec sentence features and use them in a simple classifier on a subset of `fetch_20newsgroups`.

In [1]:
import nltk
nltk.download("gutenberg", quiet=True)
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
nltk.download("stopwords", quiet=True)
print("NLTK resources ready.")

NLTK resources ready.


## Training corpus

Use `nltk.corpus.gutenberg`'s *Alice in Wonderland* text, tokenized into sentences of lowercase word tokens, as a modest built-in corpus for Word2Vec.

In [2]:
from nltk.corpus import gutenberg
from nltk.tokenize import word_tokenize

raw_text = gutenberg.raw("carroll-alice.txt")
sentences_raw = nltk.sent_tokenize(raw_text)

tokenized_corpus = [
    [w.lower() for w in word_tokenize(s) if w.isalpha()]
    for s in sentences_raw
]
tokenized_corpus = [s for s in tokenized_corpus if len(s) >= 3]

print(f"Number of sentences: {len(tokenized_corpus)}")
print("Example sentence tokens:", tokenized_corpus[10])

Number of sentences: 1491
Example sentence tokens: ['she', 'took', 'down', 'a', 'jar', 'from', 'one', 'of', 'the', 'shelves', 'as', 'she', 'passed', 'it', 'was', 'labelled', 'orange', 'marmalade', 'but', 'to', 'her', 'great', 'disappointment', 'it', 'was', 'empty', 'she', 'did', 'not', 'like', 'to', 'drop', 'the', 'jar', 'for', 'fear', 'of', 'killing', 'somebody', 'so', 'managed', 'to', 'put', 'it', 'into', 'one', 'of', 'the', 'cupboards', 'as', 'she', 'fell', 'past', 'it']


## Train Word2Vec: CBOW (`sg=0`) and Skip-Gram (`sg=1`)

In [3]:
from gensim.models import Word2Vec

w2v_cbow = Word2Vec(
    sentences=tokenized_corpus, vector_size=100, window=5, min_count=3, sg=0, epochs=20, seed=42
)
w2v_skipgram = Word2Vec(
    sentences=tokenized_corpus, vector_size=100, window=5, min_count=3, sg=1, epochs=20, seed=42
)

print(f"CBOW vocabulary size: {len(w2v_cbow.wv.key_to_index)}")
print(f"Skip-Gram vocabulary size: {len(w2v_skipgram.wv.key_to_index)}")

CBOW vocabulary size: 1014
Skip-Gram vocabulary size: 1014


## Compare `most_similar` results between CBOW and Skip-Gram

In [4]:
query_words = ["alice", "queen"]

for word in query_words:
    if word in w2v_cbow.wv.key_to_index:
        print(f"--- Most similar to '{word}' (CBOW) ---")
        for sim_word, score in w2v_cbow.wv.most_similar(word, topn=5):
            print(f"  {sim_word:15s} {score:.3f}")
    if word in w2v_skipgram.wv.key_to_index:
        print(f"--- Most similar to '{word}' (Skip-Gram) ---")
        for sim_word, score in w2v_skipgram.wv.most_similar(word, topn=5):
            print(f"  {sim_word:15s} {score:.3f}")
    print()

--- Most similar to 'alice' (CBOW) ---
  herself         0.980
  very            0.975
  quite           0.967
  but             0.966
  so              0.966
--- Most similar to 'alice' (Skip-Gram) ---
  rather          0.774
  feeling         0.766
  certainly       0.763
  timidly         0.757
  cautiously      0.753

--- Most similar to 'queen' (CBOW) ---
  with            0.982
  his             0.981
  shrill          0.980
  voice           0.980
  rabbit          0.980
--- Most similar to 'queen' (Skip-Gram) ---
  executioner     0.866
  king            0.846
  knave           0.835
  shouted         0.833
  hearts          0.824



Skip-Gram typically produces more semantically coherent neighbours on a small corpus like this because it generates far more (center word, context word) training pairs per sentence, giving rarer words more gradient updates, whereas CBOW's context-averaging trains faster but needs more data to specialise.

## Average Word2Vec sentence vectors

Represent a document as the mean of its in-vocabulary word vectors, using the CBOW model.

In [5]:
import numpy as np

def avg_word2vec(tokens, model, vector_size=100):
    vectors = [model.wv[w] for w in tokens if w in model.wv.key_to_index]
    if not vectors:
        return np.zeros(vector_size)
    return np.mean(vectors, axis=0)

example_vec = avg_word2vec(tokenized_corpus[10], w2v_cbow)
print(f"Sentence: {tokenized_corpus[10]}")
print(f"AvgWord2Vec vector shape: {example_vec.shape}")
print(example_vec[:10])

Sentence: ['she', 'took', 'down', 'a', 'jar', 'from', 'one', 'of', 'the', 'shelves', 'as', 'she', 'passed', 'it', 'was', 'labelled', 'orange', 'marmalade', 'but', 'to', 'her', 'great', 'disappointment', 'it', 'was', 'empty', 'she', 'did', 'not', 'like', 'to', 'drop', 'the', 'jar', 'for', 'fear', 'of', 'killing', 'somebody', 'so', 'managed', 'to', 'put', 'it', 'into', 'one', 'of', 'the', 'cupboards', 'as', 'she', 'fell', 'past', 'it']
AvgWord2Vec vector shape: (100,)
[ 0.30512136  0.16827916 -0.12654297 -0.02481552 -0.12899561  0.07515693
  0.22603123  0.2893484  -0.31451097  0.03689147]


## AvgWord2Vec as features for a classifier

Train a fresh Word2Vec model on a small labelled subset of `fetch_20newsgroups` (2 categories), build AvgWord2Vec features for each document, and train a `LogisticRegression` classifier on top.

In [6]:
from sklearn.datasets import fetch_20newsgroups

categories = ["rec.sport.hockey", "sci.space"]
news = fetch_20newsgroups(subset="train", categories=categories, remove=("headers", "footers", "quotes"))
news_test = fetch_20newsgroups(subset="test", categories=categories, remove=("headers", "footers", "quotes"))

print(f"Train docs: {len(news.data)}, Test docs: {len(news_test.data)}")
print(f"Categories: {news.target_names}")

Train docs: 1193, Test docs: 793
Categories: ['rec.sport.hockey', 'sci.space']


In [7]:
from nltk.corpus import stopwords

stop_words = set(stopwords.words("english"))

def tokenize_doc(doc):
    return [w.lower() for w in word_tokenize(doc) if w.isalpha() and w.lower() not in stop_words]

train_tokens = [tokenize_doc(d) for d in news.data]
test_tokens = [tokenize_doc(d) for d in news_test.data]

print("Example tokenized doc:", train_tokens[0][:15])

Example tokenized doc: ['individual', 'leaders', 'total', 'points', 'final', 'standings', 'note', 'games', 'played', 'points', 'per', 'games', 'accurate', 'player', 'team']


In [8]:
# Train Word2Vec on the training documents themselves (a modest, task-specific corpus)
news_w2v = Word2Vec(sentences=train_tokens, vector_size=100, window=5, min_count=2, sg=1, epochs=20, seed=42)
print(f"News Word2Vec vocabulary size: {len(news_w2v.wv.key_to_index)}")

News Word2Vec vocabulary size: 9141


In [9]:
X_train = np.array([avg_word2vec(toks, news_w2v) for toks in train_tokens])
X_test = np.array([avg_word2vec(toks, news_w2v) for toks in test_tokens])
y_train, y_test = news.target, news_test.target

print(f"X_train shape: {X_train.shape}, X_test shape: {X_test.shape}")

X_train shape: (1193, 100), X_test shape: (793, 100)


In [10]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

clf = LogisticRegression(max_iter=1000)
clf.fit(X_train, y_train)

y_pred = clf.predict(X_test)
acc = accuracy_score(y_test, y_pred)
print(f"Test accuracy using AvgWord2Vec features: {acc:.3f}\n")
print(classification_report(y_test, y_pred, target_names=news.target_names))

Test accuracy using AvgWord2Vec features: 0.952

                  precision    recall  f1-score   support

rec.sport.hockey       0.97      0.93      0.95       399
       sci.space       0.93      0.97      0.95       394

        accuracy                           0.95       793
       macro avg       0.95      0.95      0.95       793
    weighted avg       0.95      0.95      0.95       793



AvgWord2Vec features, despite being a very simple aggregation, give the logistic regression classifier enough signal to separate the two newsgroup topics well above chance — a dense, low-dimensional alternative to sparse BOW/TF-IDF features from topic 02.

## From-scratch: tiny NumPy CBOW forward pass

Trace one CBOW forward pass by hand on a 5-word toy vocabulary built from the sentence `"the cat sat on mat"`, with a fixed, small embedding matrix so every number is inspectable: one-hot context words &rarr; embedding lookup &rarr; average &rarr; score against every vocabulary word &rarr; softmax. Window size 1, center word `"sat"`, context words `["cat", "on"]`.

In [11]:
import numpy as np

toy_vocab = ["the", "cat", "sat", "on", "mat"]
word_to_idx = {w: i for i, w in enumerate(toy_vocab)}
V = len(toy_vocab)  # vocabulary size
d = 3               # embedding dimensionality

def one_hot(word):
    vec = np.zeros(V)
    vec[word_to_idx[word]] = 1.0
    return vec

context_words = ["cat", "on"]
center_word = "sat"

context_one_hots = np.array([one_hot(w) for w in context_words])
print("Context one-hot vectors (rows = 'cat', 'on'):")
print(context_one_hots)

Context one-hot vectors (rows = 'cat', 'on'):
[[0. 1. 0. 0. 0.]
 [0. 0. 0. 1. 0.]]


### Step 1: fixed input/output embedding matrices

Small, hand-chosen values (not randomly initialized) so the arithmetic below is traceable by hand. In real Word2Vec training these start random and are *learned* by gradient descent; here they're fixed constants to inspect one forward pass in isolation.

In [12]:
# V_in: input (context) embeddings, one row per vocabulary word -> shape (V, d)
V_in = np.array([
    [0.1, 0.2, 0.0],   # the
    [0.9, 0.1, 0.0],   # cat
    [0.0, 0.0, 0.0],   # sat  (this is the word we're trying to predict -- unused as input here)
    [0.8, 0.0, 0.1],   # on
    [0.2, 0.7, 0.1],   # mat
])

# V_out: output (center-word scoring) embeddings -> shape (d, V)
V_out = np.array([
    [0.1, 0.4, 0.7, 0.2, 0.3],
    [0.5, 0.2, 0.6, 0.1, 0.4],
    [0.3, 0.1, 0.5, 0.6, 0.2],
])

print("V_in shape:", V_in.shape, " V_out shape:", V_out.shape)

V_in shape: (5, 3)  V_out shape: (3, 5)


### Step 2: embedding lookup + average -> hidden vector `h`

One-hot-encoding a context word and multiplying by `V_in` is exactly a row lookup: `one_hot(w) @ V_in == V_in[word_to_idx[w]]`. CBOW averages the looked-up context embeddings into a single hidden vector `h`.

In [13]:
# Embedding lookup via one-hot @ V_in (equivalent to V_in[word_to_idx[w]])
context_embeddings = context_one_hots @ V_in
print("Looked-up context embeddings ('cat', 'on'):")
print(context_embeddings)

# CBOW hidden vector: average of context embeddings
h = context_embeddings.mean(axis=0)
print("\nHidden vector h (average of context embeddings):", h)

Looked-up context embeddings ('cat', 'on'):
[[0.9 0.1 0. ]
 [0.8 0.  0.1]]

Hidden vector h (average of context embeddings): [0.85 0.05 0.05]


### Step 3: score every vocabulary word, then softmax

`z = h @ V_out` produces one score per vocabulary word (how well each word matches the averaged context). Softmax turns those scores into a probability distribution over the whole vocabulary -- this is `P(center word | context)`.

In [14]:
z = h @ V_out  # score for every vocabulary word
print("Raw scores z (one per vocab word):", z)

def softmax(scores):
    exp_scores = np.exp(scores - scores.max())  # subtract max for numerical stability
    return exp_scores / exp_scores.sum()

probs = softmax(z)
print("\nSoftmax probabilities over the vocabulary:")
for word, p in zip(toy_vocab, probs):
    marker = "  <-- true center word" if word == center_word else ""
    print(f"  P({word!r:6s}) = {p:.4f}{marker}")

true_idx = word_to_idx[center_word]
loss = -np.log(probs[true_idx])
print(f"\nCross-entropy loss for true center word {center_word!r}: {loss:.4f}")

Raw scores z (one per vocab word): [0.125 0.355 0.65  0.205 0.285]

Softmax probabilities over the vocabulary:
  P('the' ) = 0.1611
  P('cat' ) = 0.2028
  P('sat' ) = 0.2724  <-- true center word
  P('on'  ) = 0.1746
  P('mat' ) = 0.1891

Cross-entropy loss for true center word 'sat': 1.3005


These hand-picked `V_in`/`V_out` weren't trained, so `"sat"` getting the top probability above is a coincidence of the chosen constants, not evidence of a trained model -- there's no gradient step in this cell, only a forward pass. Real Word2Vec training repeats this exact forward pass (one-hot context &rarr; lookup &rarr; average &rarr; score &rarr; softmax) over every window in the corpus, computes the gradient of this same cross-entropy loss with respect to `V_in` and `V_out`, and takes a gradient-descent step after each one (or each mini-batch, with negative sampling replacing the full softmax for efficiency) -- millions of repetitions of exactly this cell, with weights *updated* each time rather than held fixed, is what turns `V_in` into `gensim`'s trained `model.wv`.